# Maximum Likelihood Estimation

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 03.06 |
| Time | ~70 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/03_Probability_and_Statistics/07_maximum_likelihood_estimation.ipynb)

---

## 🎯 Learning Objective

Derive maximum likelihood estimation from first principles and connect it directly to the cross-entropy loss used to train neural networks.

---

## 📐 Theory

Every time you train a model by minimizing a loss function, you are almost certainly doing
maximum likelihood estimation, whether or not you've framed it that way. This notebook makes
that connection explicit and precise.

### The likelihood function

Given data $D=\{x_1,\ldots,x_n\}$ assumed drawn i.i.d. from a distribution with unknown
parameter $\theta$, the **likelihood** is the joint probability of the data, viewed as a
function of $\theta$ with the data held fixed:

$$L(\theta) = P(D\mid\theta) = \prod_{i=1}^n p(x_i\mid\theta),$$

the product coming directly from independence (`03.04`). This is the *same* object that
appears in the numerator of Bayes' theorem (`03.05`) — MLE is exactly MAP estimation with a
flat (uninformative) prior, so whatever $\theta$ makes the observed data most probable is
declared the estimate:

$$\hat\theta_{\text{MLE}} = \arg\max_\theta L(\theta) = \arg\max_\theta \prod_{i=1}^n p(x_i\mid\theta).$$

### Log-likelihood: why we always take the log

Products of many small probabilities underflow numerically and are painful to differentiate.
The logarithm is strictly increasing, so it never changes *where* the maximum is, while turning
the product into a sum:

$$\ell(\theta) = \log L(\theta) = \sum_{i=1}^n \log p(x_i\mid\theta), \qquad \hat\theta_{\text{MLE}} = \arg\max_\theta \ell(\theta).$$

Setting $\nabla_\theta \ell(\theta) = 0$ and solving (or, when there's no closed form, running
gradient *ascent* on $\ell$, or equivalently gradient *descent* on $-\ell$) is how MLE is
actually computed in practice — precisely the mechanics of `02.05`
(link: [Gradients and Directional Derivatives](../02_Calculus/05_gradients_and_directional_derivatives.ipynb)).

### Cross-entropy loss IS negative log-likelihood

For a classifier producing $\hat p_i = P(\text{class}=y_i\mid x_i;\theta)$ (a softmax output,
`03.02`) on a labeled dataset, the log-likelihood is $\ell(\theta) = \sum_i \log \hat p_i$. The
**cross-entropy loss** used to train the network is defined as
$\mathcal{L}_{\text{CE}}(\theta) = -\frac{1}{n}\sum_i \log \hat p_i$ — literally $-\ell(\theta)/n$.
Therefore:

$$\arg\min_\theta \mathcal{L}_{\text{CE}}(\theta) = \arg\min_\theta \left(-\frac{1}{n}\ell(\theta)\right) = \arg\max_\theta \ell(\theta) = \hat\theta_{\text{MLE}}.$$

Minimizing cross-entropy and maximizing likelihood are the *same optimization problem*, up to
the sign flip and the constant $1/n$ that don't change where the optimum is. This equivalence
is not an analogy — it's an algebraic identity, demonstrated numerically below.

### Fisher information and asymptotic properties

The **Fisher information** $I(\theta) = \mathbb{E}\left[\left(\frac{\partial}{\partial\theta}
\log p(X\mid\theta)\right)^2\right] = -\mathbb{E}\left[\frac{\partial^2}{\partial\theta^2}\log
p(X\mid\theta)\right]$ measures how sharply the log-likelihood curves at the true $\theta$: high
Fisher information means a sharp peak, so data is highly informative about $\theta$. Under
standard regularity conditions, the MLE is **consistent** ($\hat\theta_{\text{MLE}} \to
\theta_{\text{true}}$ as $n\to\infty$) and **asymptotically normal**,
$\hat\theta_{\text{MLE}} \approx \mathcal{N}(\theta_{\text{true}}, 1/(nI(\theta)))$ — variance
shrinks as $1/n$, scaled inversely by how informative each observation is.

### The EM algorithm

When some data is unobserved — a **latent variable** $z_i$ per data point, as in a mixture
model — the log-likelihood $\log\sum_z p(x,z\mid\theta)$ involves a log of a sum, which usually
has no closed-form maximum. The **Expectation-Maximization (EM) algorithm** breaks this into
two tractable steps, iterated to convergence: the **E-step** computes the posterior
responsibility $P(z_i\mid x_i,\theta^{(t)})$ of each latent value under the current parameters
(Bayes' theorem again), and the **M-step** maximizes the *expected complete-data
log-likelihood* under those responsibilities — now a log of individual terms, so it usually has
a closed form. Each EM iteration provably never decreases the observed log-likelihood, and it's
the backbone of fitting the Gaussian Mixture Models in
[Probabilistic Models in Machine Learning](10_probabilistic_models_in_ml.ipynb) (`03.10`).

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import torch
import torch.nn as nn
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Plotting the log-likelihood of a Gaussian's mean $\mu$ against candidate values, side by side
with the negative log-likelihood, makes "maximizing likelihood = minimizing negative
log-likelihood" visually obvious: one curve is a vertical flip of the other, and they peak/dip
at the exact same $\mu$.

In [ ]:
# Log-likelihood vs. negative log-likelihood for a Gaussian's mean mu, with sigma known/fixed --
# one is the exact vertical flip of the other, so their optimizers coincide.
rng = np.random.default_rng(0)
true_mu, sigma = 3.0, 1.5
data = rng.normal(true_mu, sigma, size=60)

mu_grid = np.linspace(0, 6, 300)
log_lik = np.array([np.sum(norm.logpdf(data, loc=mu, scale=sigma)) for mu in mu_grid])
neg_log_lik = -log_lik

mle_mu_numeric = mu_grid[np.argmax(log_lik)]
mle_mu_closed_form = data.mean()   # for a Gaussian with known sigma, MLE of mu IS the sample mean

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(mu_grid, log_lik, color="#4C72B0", lw=2)
axes[0].axvline(mle_mu_closed_form, color="red", ls="--", label=f"argmax = {mle_mu_closed_form:.3f}")
axes[0].set_title("Log-likelihood ℓ(μ)  [maximize this]")
axes[0].set_xlabel("μ"); axes[0].set_ylabel("ℓ(μ)"); axes[0].legend()

axes[1].plot(mu_grid, neg_log_lik, color="#C44E52", lw=2)
axes[1].axvline(mle_mu_closed_form, color="red", ls="--", label=f"argmin = {mle_mu_closed_form:.3f}")
axes[1].set_title("Negative log-likelihood -ℓ(μ)  [minimize this]")
axes[1].set_xlabel("μ"); axes[1].set_ylabel("-ℓ(μ)"); axes[1].legend()
plt.tight_layout()
plt.show()

print(f"argmax of log-likelihood (grid search):      mu = {mle_mu_numeric:.4f}")
print(f"argmin of negative log-likelihood (grid search): mu = {mu_grid[np.argmin(neg_log_lik)]:.4f}")
print(f"Closed-form MLE (sample mean):                 mu = {mle_mu_closed_form:.4f}")
print(f"True mu used to generate the data:              mu = {true_mu}")
print("All three agree (up to grid resolution) -- maximizing L and minimizing -L pick the same point.")

## 🐍 Implementation from Scratch

We derive and implement the closed-form MLE for a Gaussian's mean and variance, verify the
Fisher-information-predicted sampling variance of the MLE via simulation, and implement the EM
algorithm from scratch for a 2-component Gaussian mixture with a latent cluster assignment.

In [ ]:
# MLE of a Gaussian's mean/variance, in closed form (found by setting d(log-lik)/d(param)=0)
def gaussian_mle(data):
    mu_hat = data.mean()                        # argmax_mu of the log-likelihood: the sample mean
    sigma2_hat = np.mean((data - mu_hat) ** 2)   # argmax_sigma2: the (biased) sample variance
    return mu_hat, sigma2_hat

rng = np.random.default_rng(1)
true_mu, true_sigma = 3.0, 2.0
sample = rng.normal(true_mu, true_sigma, size=500)
mu_hat, sigma2_hat = gaussian_mle(sample)
print(f"Our MLE: mu_hat={mu_hat:.4f} (true {true_mu}), sigma_hat={np.sqrt(sigma2_hat):.4f} (true {true_sigma})")

scipy_mu, scipy_sigma = norm.fit(sample)  # independent cross-check via scipy's own MLE fitting
print(f"scipy.stats.norm.fit: mu={scipy_mu:.4f}, sigma={scipy_sigma:.4f}  (matches our closed form)")

# Fisher information for a Gaussian mean (sigma known) is I(mu) = 1/sigma^2 analytically.
# Confirm the Cramer-Rao/asymptotic-normality prediction Var(MLE) ~ 1/(n*I(mu)) = sigma^2/n
# by literally repeating the "collect n samples, compute the MLE" experiment thousands of times.
n_per_experiment, n_experiments = 40, 5000
mle_estimates = np.array([
    rng.normal(true_mu, true_sigma, size=n_per_experiment).mean() for _ in range(n_experiments)
])
empirical_var = mle_estimates.var()
fisher_info_mu = 1 / true_sigma ** 2
predicted_var = 1 / (n_per_experiment * fisher_info_mu)
print(f"\nEmpirical variance of the MLE across {n_experiments} repeated experiments (n={n_per_experiment} each): {empirical_var:.5f}")
print(f"Fisher-information-predicted variance sigma^2/n = 1/(n*I(mu)):                                {predicted_var:.5f}")

# EM from scratch: fit a 2-component 1D Gaussian mixture where cluster membership is a LATENT
# variable never directly observed -- only the mixed-together x values are seen.
rng2 = np.random.default_rng(2)
true_means, true_stds, true_weights = np.array([-2.0, 3.0]), np.array([1.0, 1.5]), np.array([0.4, 0.6])
component = rng2.choice(2, size=1000, p=true_weights)
data_gmm = rng2.normal(true_means[component], true_stds[component])

means, stds, weights = np.array([0.0, 1.0]), np.array([1.0, 1.0]), np.array([0.5, 0.5])  # deliberately off
log_likelihoods = []
for _ in range(50):
    # E-step: responsibility r_ik = P(z_i=k | x_i, theta_current) -- Bayes' theorem (03.05) applied
    # to "which component generated this point", using the CURRENT parameter guess
    comp_likelihoods = np.array([weights[k] * norm(means[k], stds[k]).pdf(data_gmm) for k in range(2)])
    total_likelihood = comp_likelihoods.sum(axis=0)
    responsibilities = comp_likelihoods / total_likelihood          # shape (2, n), columns sum to 1
    log_likelihoods.append(np.sum(np.log(total_likelihood)))         # observed-data log-likelihood

    # M-step: maximize the expected complete-data log-likelihood -- now a weighted Gaussian MLE,
    # closed-form because responsibilities (not raw counts) act as soft cluster memberships
    Nk = responsibilities.sum(axis=1)
    weights = Nk / len(data_gmm)
    means = (responsibilities @ data_gmm) / Nk
    stds = np.sqrt((responsibilities * (data_gmm - means[:, None]) ** 2).sum(axis=1) / Nk)

order = np.argsort(means)  # sort recovered components by mean to align with true_means (already sorted)
print(f"\nEM recovered means:   {np.round(means[order], 3)}  (true: {true_means})")
print(f"EM recovered stds:    {np.round(stds[order], 3)}  (true: {true_stds})")
print(f"EM recovered weights: {np.round(weights[order], 3)}  (true: {true_weights})")

diffs = np.diff(log_likelihoods)
print(f"\nLog-likelihood monotonically non-decreasing across all EM iterations? {bool(np.all(diffs >= -1e-8))}")
print(f"Log-likelihood: {log_likelihoods[0]:.2f} -> {log_likelihoods[-1]:.2f}  (EM's defining guarantee)")

## 🤖 Why This Matters for AI

Training a neural network with cross-entropy loss **is** maximum likelihood estimation:
cross-entropy is, by direct algebraic identity, the negative log-likelihood of the labels under
the model's predicted distribution, averaged over the batch. This isn't a loose analogy — the
cell below computes both quantities, at every training step, on an actual model trained with a
real optimizer, and shows they are the same number up to the sign flip and the $1/n$
normalization the Theory section derived.

---

In [ ]:
# THE central claim of this notebook, shown numerically: training a logistic-regression
# classifier by minimizing cross-entropy loss IS maximizing likelihood -- not similar, IDENTICAL.
rng = np.random.default_rng(0)
torch.manual_seed(0)

n_samples, n_features = 300, 4
X_np = rng.normal(size=(n_samples, n_features))
true_w = np.array([1.5, -2.0, 0.5, 1.0])
probs_true = 1 / (1 + np.exp(-(X_np @ true_w)))
y_np = (rng.uniform(size=n_samples) < probs_true).astype(np.float32)

X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.float32)

model = nn.Linear(n_features, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
bce_loss_fn = nn.BCEWithLogitsLoss(reduction="mean")  # THE cross-entropy loss, for 2 classes

ce_losses, neg_ll_over_n = [], []
print(f"{'step':>5} | {'cross-entropy loss':>19} | {'log-likelihood':>15} | {'accuracy':>8}")
for step in range(151):
    logits = model(X).squeeze(-1)
    loss = bce_loss_fn(logits, y)  # what the optimizer actually minimizes

    with torch.no_grad():
        probs = torch.sigmoid(logits)
        # the SAME quantity computed independently, straight from this notebook's own
        # log-likelihood definition ell(theta) = sum_i log p(y_i | x_i; theta)
        log_likelihood = torch.sum(y * torch.log(probs) + (1 - y) * torch.log(1 - probs)).item()
        accuracy = ((probs > 0.5).float() == y).float().mean().item()
    ce_losses.append(loss.item())
    neg_ll_over_n.append(-log_likelihood / n_samples)

    if step % 30 == 0:
        print(f"{step:5d} | {loss.item():19.6f} | {log_likelihood:15.4f} | {accuracy:8.3f}")

    optimizer.zero_grad()
    loss.backward()   # the TRUE gradient of the cross-entropy loss -- no heuristic update rule
    optimizer.step()

max_diff = np.max(np.abs(np.array(ce_losses) - np.array(neg_ll_over_n)))
print(f"\nmax |cross-entropy loss - (-log-likelihood / n)| across all {len(ce_losses)} steps: {max_diff:.2e}")
print("Zero up to floating point -- minimizing cross-entropy loss and maximizing the log-likelihood")
print("are the exact same optimization problem, confirmed at every single step of real training.")

## 🏋️ Exercises

### Warm-up
1. For a Bernoulli($p$) likelihood with $k$ successes out of $n$ trials, derive
   $\hat{p}_{\text{MLE}}$ by setting $\frac{d\ell}{dp}=0$ by hand. Verify it equals $k/n$ using
   the `gaussian_mle`-style pattern above (write a `bernoulli_mle` function and test it).

### Practice
2. In the Fisher information cell, repeat the experiment for `n_per_experiment` in `[10, 40,
   160, 640]`. Plot empirical variance vs. $n$ on a log-log plot alongside the predicted
   $\sigma^2/n$ curve, and confirm they track each other across all four values.

### Challenge
3. Modify the cross-entropy/log-likelihood training loop to use a *heuristic* update instead of
   `loss.backward()` — e.g. update each weight by a fixed step in the direction of
   `sign(gradient)` rather than the gradient itself. Rerun and check whether `max_diff` between
   cross-entropy loss and $-\text{log-likelihood}/n$ still holds (it should — the identity is
   about the *loss function*, not the optimizer), but explain why training now converges
   differently. Then connect this to why Adam/SGD, not ad-hoc heuristics, are used to actually
   climb the likelihood surface in practice.

---

## 📚 Further Reading
- Casella & Berger, *Statistical Inference*, Ch. 7 (Point Estimation) and Ch. 10 (Asymptotic Evaluations)
- Dempster, Laird & Rubin, ["Maximum Likelihood from Incomplete Data via the EM Algorithm"](https://www.jstor.org/stable/2984875) (1977)
- Goodfellow, Bengio & Courville, *Deep Learning*, Ch. 5.5 (Maximum Likelihood Estimation)

---

*Next notebook: [Hypothesis Testing and Confidence Intervals](08_hypothesis_testing_confidence.ipynb)*